In [1]:
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
songs = joblib.load("../data/processed/songs.pkl")

audio_matrix = np.load("../data/processed/audio_matrix.npy")

tfidf_matrix = sparse.load_npz("../data/processed/tfidf_matrix.npz")

vectorizer = joblib.load("../data/processed/tfidf_vectorizer.pkl")

In [3]:
print(songs.shape)
print(audio_matrix.shape)
print(tfidf_matrix.shape)

(498052, 15)
(498052, 9)
(498052, 35354)


In [4]:
user_tracks = [
    {
        "song": "Hold My Liquor",
        "artist": "Kanye West"
    },
    {
        "song": "Dogs",
        "artist": "Pink Floyd"
    },
    {
        "song": "Drugs You Should Try It",
        "artist": "Travis Scott"
    },
    {
        "song": "Comfortably Numb",
        "artist": "Pink Floyd"
    },
    {
        "song": "Hotel California",
        "artist": "Eagles"
    }
]

In [13]:
favorite_genres = [
    "rock",
    "alternative rock",
    "hip hop",
    "rap"
]

favorite_artists = [
    "Kanye West",
    "Travis Scott",
    "Pink Floyd",
    "King Crimson"
]

In [14]:
metadata_query = " ".join(
    favorite_artists + favorite_genres
)

metadata_query

'Kanye West Travis Scott Pink Floyd King Crimson rock alternative rock hip hop rap'

In [8]:
matched_rows = []

for track in user_tracks:
    match = songs[(songs["song"].str.lower() == track["song"].lower()) & (songs["Artist(s)"].str.contains(track["artist"], case=False, na=False))]

    if not match.empty:
        matched_rows.append(match)

matched = pd.concat(matched_rows, ignore_index=False)

matched

,song,Artist(s),Genre,Album,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Tempo,Loudness (db),metadata
225271,Hold My Liquor,Kanye West,"synthpop,pop,rap",Yeezus,61,32,50,4,3,12,2,0,107,-10.36,"Kanye West synthpop,pop,rap"
335147,Dogs,Pink Floyd,"rock,progressive rock,hard rock",Animals,58,48,32,14,5,15,8,0,109,-11.40,"Pink Floyd rock,progressive rock,hard rock"
458911,Drugs You Should Try It,Travis Scott,"rap,hip hop,trap",Birds In The Trap Sing McKnight,79,59,51,59,6,12,60,5,108,-8.15,"Travis Scott rap,hip hop,trap"
335138,Comfortably Numb,Pink Floyd,"rock,progressive rock,hard rock",The Wall,72,37,47,17,3,8,15,31,127,-12.60,"Pink Floyd rock,progressive rock,hard rock"
126437,Hotel California,Eagles,"rock,pop,classic rock",Hotel California (2013 Remaster),83,51,58,61,3,6,1,0,147,-9.48,"Eagles rock,pop,classic rock"


In [9]:
indices = matched.index.to_numpy()

indices

array([225271, 335147, 458911, 335138, 126437], dtype=int64)

In [10]:
user_audio_vectors = audio_matrix[indices]

user_audio_vectors.shape

(5, 9)

In [12]:
user_audio_profile = np.mean(user_audio_vectors, axis=0)

user_audio_profile.shape

(9,)

In [15]:
user_metadata_vector = vectorizer.transform([metadata_query])

In [16]:
audio_similarity = cosine_similarity(
    user_audio_profile.reshape(1, -1),
    audio_matrix
).flatten()

audio_similarity

array([-0.48104882, -0.60027568, -0.60009766, ..., -0.03318184,
       -0.40349554,  0.0695393 ])

In [17]:
metadata_similarity = cosine_similarity(
    user_metadata_vector,
    tfidf_matrix
).flatten()

metadata_similarity

array([0.1042796, 0.1042796, 0.1042796, ..., 0.1042796, 0.1042796,
       0.1042796])

In [30]:
alpha = 0.6
beta = 0.4

final_scores = alpha * audio_similarity + beta * metadata_similarity
final_scores[indices] = -1
final_scores

array([-0.24691745, -0.31845357, -0.31834676, ...,  0.02180273,
       -0.20038549,  0.08343542])

In [31]:
top_indices = np.argsort(final_scores)[-20:][::-1]

recommendations = songs.iloc[top_indices].copy()

recommendations["score"] = final_scores[top_indices]
recommendations["audio_score"] = audio_similarity[top_indices]
recommendations["metadata_score"] = metadata_similarity[top_indices]
recommendations[["song", "Artist(s)", "Genre", "score" , "audio_score" , "metadata_score"]]

,song,Artist(s),Genre,score,audio_score,metadata_score
235312,Epitaph,King Crimson,"rock,progressive rock,jazz",0.753314,0.942120,0.470105
335236,Burning Bridges,Pink Floyd,"rock,progressive rock,hard rock",0.742451,0.920672,0.475120
335249,The Narrow Way,Pink Floyd,"rock,progressive rock,hard rock",0.731978,0.903216,0.475120
235334,The Night Watch,King Crimson,"rock,progressive rock,jazz",0.719587,0.885908,0.470105
335311,On The Turning Away,Pink Floyd,"rock,progressive rock,hard rock",0.719369,0.882201,0.475120
335268,The Narrow Way: Part 3,Pink Floyd,"rock,progressive rock,hard rock",0.719369,0.882201,0.475120
335193,On the Turning Away,Pink Floyd,"rock,progressive rock,hard rock",0.719369,0.882201,0.475120
335260,The Doctor Comfortably Numb,Pink Floyd,"rock,progressive rock,hard rock",0.719211,0.881939,0.475120
335351,Wined And Dined,Pink Floyd,"rock,progressive rock,hard rock",0.718017,0.879949,0.475120
335271,Matilda mother - alternative version,Pink Floyd,"rock,progressive rock,hard rock",0.715425,0.875629,0.475120
